# Computer-Assisted Proof: Lie Algebra Generation for Lorenz 96 Model (Gap 4 Forcing)

**Context:** This notebook implements the revised computer-assisted proof (CAP) for the paper *"Non-uniqueness of stationary measures for stochastic systems with almost surely invariant manifolds"*. Following the discovery of a resonance bug with $3\mathbb{Z}/N\mathbb{Z}$ forcing, this notebook calculates the Lie algebra generation under $4\mathbb{Z}/N\mathbb{Z}$ forcing (gap of 4) to verify that the algebraic generation of the transverse special linear algebra holds without the destructive resonance.

**Objective:** The primary goal is to verify that the Lie algebra generated by the matrices $M_k = DB(e_k)|_{H_I^\perp}$ is the special linear Lie algebra $\mathfrak{sl}(H_I^\perp)$.

**Methodology:**
1.  **Matrix Representation:** Define the exact structural matrices $M_k$ using Sympy, with the corrected modulation arithmetic indices that accurately reflecting the sum $M_k = E_{k-1,k-2} - E_{k+2,k+1} + E_{k+1,k+2} - E_{k+1,k-1}$.
2.  **Lie Brackets:** Compute iterated Lie brackets $[M_k, M_l]$ up to depth 4.
3.  **Span Verification:** Use symbolic linear algebra (Reduced Row Echelon Form) on the vectorized matrices to determine the basis of the spanned space.
4.  **Elementary Matrix Check:** Identify if elementary matrices are present, which combined with shift-invariance generates all of $\mathfrak{sl}(H_I^\perp)$.

**Implementation Details:**
*   The calculations are performed for $N=12$ with gap 4 (transverse dimension 9).

In [1]:
%pip install sympy numpy matplotlib > /dev/null 2>&1

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sympy as sp
from sympy import Matrix, zeros, Rational
from IPython.display import display, Math

sp.init_printing()  # Use exact arithmetic


## Matrix Generation

First, we implement the L96 matrices. This uses the mathematically correct indices for the $B(u,v)$ transverse Jacobian.

In [3]:
def discrete_delta(i, j):
    """1D delta function for modulo arithmetic"""
    return Rational(1 if i == j else 0)

def B_coefficient(k, j, l, n):
    """Matrix coefficient exactly mapping to M_k = E_{k-1,k-2} - E_{k+2,k+1} + E_{k+1,k+2} - E_{k+1,k-1}"""
    # Note we use modulo n arithmetic to wrap correctly
    delta1 = discrete_delta(j % n, (k-1) % n) * discrete_delta(l % n, (k-2) % n)
    delta2 = discrete_delta(j % n, (k+2) % n) * discrete_delta(l % n, (k+1) % n)
    delta3 = discrete_delta(j % n, (k+1) % n) * discrete_delta(l % n, (k+2) % n)
    delta4 = discrete_delta(j % n, (k+1) % n) * discrete_delta(l % n, (k-1) % n)
    
    return delta1 - delta2 + delta3 - delta4

def get_transverse_indices(n, gap=4):
    """Get indices not divisible by the gap"""
    return [i for i in range(n) if i % gap != 0]

def generate_h_matrix(k, trans_index, n):
    """Generate matrix for index k."""
    size = len(trans_index)
    H = zeros(size, size)
    
    for i, j in enumerate(trans_index):
        for m, l in enumerate(trans_index):
            H[i, m] = B_coefficient(k, j, l, n)
            
    return H

def compute_bracket(A, B):
    """Compute [A,B] using rational arithmetic."""
    return A * B - B * A

# Generate matrices for Gap 4
n = 12 
gap = 4
depth = 4 # Maximum depth for bracket computation
trans_index = get_transverse_indices(n, gap)
h_indices = [4, 8, 12] # Indices for forced modes

print(f"System size n = {n}")
print(f"Gap size = {gap}")
print(f"Transverse dimensions = {len(trans_index)}")

h_matrices = [generate_h_matrix(k, trans_index, n) for k in h_indices]
# Print generated basic transverse matrices
for idx, H in enumerate(h_matrices):
    display(Math(f"M_{{{h_indices[idx]}}} ="))
    display(H)


System size n = 12
Gap size = 4
Transverse dimensions = 9


<IPython.core.display.Math object>

⎡0  0  0   0   0  0  0  0  0⎤
⎢                           ⎥
⎢0  0  0   0   0  0  0  0  0⎥
⎢                           ⎥
⎢0  1  0   0   0  0  0  0  0⎥
⎢                           ⎥
⎢0  0  -1  0   1  0  0  0  0⎥
⎢                           ⎥
⎢0  0  0   -1  0  0  0  0  0⎥
⎢                           ⎥
⎢0  0  0   0   0  0  0  0  0⎥
⎢                           ⎥
⎢0  0  0   0   0  0  0  0  0⎥
⎢                           ⎥
⎢0  0  0   0   0  0  0  0  0⎥
⎢                           ⎥
⎣0  0  0   0   0  0  0  0  0⎦

<IPython.core.display.Math object>

⎡0  0  0  0  0  0   0   0  0⎤
⎢                           ⎥
⎢0  0  0  0  0  0   0   0  0⎥
⎢                           ⎥
⎢0  0  0  0  0  0   0   0  0⎥
⎢                           ⎥
⎢0  0  0  0  0  0   0   0  0⎥
⎢                           ⎥
⎢0  0  0  0  0  0   0   0  0⎥
⎢                           ⎥
⎢0  0  0  0  1  0   0   0  0⎥
⎢                           ⎥
⎢0  0  0  0  0  -1  0   1  0⎥
⎢                           ⎥
⎢0  0  0  0  0  0   -1  0  0⎥
⎢                           ⎥
⎣0  0  0  0  0  0   0   0  0⎦

<IPython.core.display.Math object>

⎡0   1  0  0  0  0  0  0  -1⎤
⎢                           ⎥
⎢-1  0  0  0  0  0  0  0  0 ⎥
⎢                           ⎥
⎢0   0  0  0  0  0  0  0  0 ⎥
⎢                           ⎥
⎢0   0  0  0  0  0  0  0  0 ⎥
⎢                           ⎥
⎢0   0  0  0  0  0  0  0  0 ⎥
⎢                           ⎥
⎢0   0  0  0  0  0  0  0  0 ⎥
⎢                           ⎥
⎢0   0  0  0  0  0  0  0  0 ⎥
⎢                           ⎥
⎢0   0  0  0  0  0  0  0  0 ⎥
⎢                           ⎥
⎣0   0  0  0  0  0  0  1  0 ⎦

## Generate and Compute Brackets

We compute all brackets up to depth 4.

In [4]:
def compute_all_brackets(h_matrices, h_indices, max_depth=4):
    """Compute all brackets up to given depth."""
    current = h_matrices.copy()
    all_matrices = current.copy()
    ops = [f"M_{k}" for k in h_indices]
    all_ops = ops.copy()
    
    print(f"Computing brackets up to depth {max_depth}...")
    
    for depth in range(2, max_depth + 1):
        print(f"\nDepth {depth}:")
        new_matrices = []
        new_ops = []
        
        # Compute new brackets against the core generators
        for i, H in enumerate(h_matrices):
            for j, M in enumerate(current):
                bracket = compute_bracket(H, M)
                if not bracket.is_zero_matrix:
                    new_matrices.append(bracket)
                    new_ops.append(f"[M_{h_indices[i]}, {ops[j]}]")
        
        if not new_matrices:
            print("No new brackets found")
            break
            
        print(f"Found {len(new_matrices)} new brackets")
        all_matrices.extend(new_matrices)
        all_ops.extend(new_ops)
        current = new_matrices
        ops = new_ops
    
    return all_matrices, all_ops

# Compute all brackets
all_matrices, all_ops = compute_all_brackets(h_matrices, h_indices, max_depth=depth)
print(f"\nTotal matrices: {len(all_matrices)}")

Computing brackets up to depth 4...

Depth 2:
Found 6 new brackets

Depth 3:
Found 12 new brackets

Depth 4:
Found 24 new brackets

Total matrices: 45


## Find Elementary Matrices

We use row reduction to find linear combinations that yield elementary matrices:

In [5]:
def find_elementary_matrices(matrices, ops):
    """Find elementary matrices using row reduction."""
    matrix_size = matrices[0].rows
    
    # Stack matrices as vectors
    stack = zeros(len(matrices), matrix_size * matrix_size)
    for i, M in enumerate(matrices):
        for r in range(matrix_size):
            for c in range(matrix_size):
                stack[i, r*matrix_size + c] = M[r,c]
    
    print(f"Matrix stack shape: {stack.rows} × {stack.cols}")
    
    # Compute RREF
    print("\nComputing row reduction...")
    rref, pivots = stack.rref()
    
    # Check each row for elementary matrices
    elementary = []
    print("\nChecking for generated elementary matrices:")
    
    for i in range(rref.rows):
        # Convert row back to matrix
        M = zeros(matrix_size, matrix_size)
        for r in range(matrix_size):
            for c in range(matrix_size):
                M[r,c] = rref[i, r*matrix_size + c]
        
        # Count non-zero entries
        non_zero = [(r, c, M[r,c]) for r in range(matrix_size)
                    for c in range(matrix_size) if M[r,c] != 0]
        
        if len(non_zero) == 1:
            r, c, val = non_zero[0]
            elementary.append((r, c, val))
            print(f"Found E_{{{r+1},{c+1}}}")
    
    return elementary

# Find elementary matrices
elementary_matrices = find_elementary_matrices(all_matrices, all_ops)

if len(elementary_matrices) > 0:
    print("\nSUCCESS! The algorithm successfully generates elementary matrices from the Lie brackets of gap 4 forcing.")
else:
    print("\nFAILURE. No elementary matrices found. Deeper brackets or a different strategy may be needed.")

Matrix stack shape: 45 × 81

Computing row reduction...

Checking for generated elementary matrices:
Found E_{1,7}
Found E_{2,7}
Found E_{3,1}
Found E_{3,8}
Found E_{4,1}
Found E_{5,1}
Found E_{6,2}
Found E_{6,4}
Found E_{7,4}
Found E_{8,4}
Found E_{9,5}
Found E_{9,7}

SUCCESS! The algorithm successfully generates elementary matrices from the Lie brackets of gap 4 forcing.
